# Étape 14 (partie B1 : graines 5, 6 et 7) — La conclusion tient-elle avec un modèle publié ? Réplication avec GKT

À l'étape 13 (PrereqKT), la **couverture** du graphe expliquait l'essentiel du bénéfice ; la **justesse** et la **proximité thématique** des flèches comptaient moins.
Un relecteur demandera : *« est-ce propre à votre modèle ? »*. On refait donc l'expérience avec **GKT** (*Graph-based Knowledge Tracing*, Nakagawa, Iwasawa et Matsuo, 2019), un modèle publié très cité.

**Comment GKT utilise le graphe.** Chaque exercice a son propre état caché (« niveau de maîtrise »). Quand l'élève répond à l'exercice *i*,
GKT met à jour l'état de *i* **et celui de ses voisins dans le graphe** (ses prérequis et ses successeurs), puis prédit la prochaine réponse à partir de l'état de l'exercice suivant.
Sans graphe, une réponse ne modifie que l'exercice répondu : le graphe est donc **le seul canal** qui transmet l'information d'un exercice à l'autre.
C'est un usage du graphe très différent de PrereqKT : un bon test de généralité.

Réimplémentation fidèle à l'article (variante à graphe fixe, avec messages distincts vers prérequis et successeurs, porte effacement-ajout et mise à jour GRU ; dimensions 32 comme dans l'article),
avec une seule adaptation : l'état initial de chaque exercice est **appris** (sinon, sans graphe, toutes les premières tentatives recevraient la même prédiction).

**Conditions** (graphes identiques à l'étape 13, mêmes graines) : `aucun`, `expert_c25`, `expert_c100`, `theme_faux_c100`, `hors_theme_faux_c100`.
**10 graines × 2 régimes (5 % et 1 % des élèves) × 5 conditions = 100 entraînements GKT.**

**Hypothèses principales, déclarées avant l'expérience** (AUC des premières tentatives ; tests t appariés unilatéraux par graine ; Holm sur les 8 tests) :
- **H1 — couverture :** `expert_c100` > `expert_c25` ;
- **H2 — justesse :** `expert_c100` > `theme_faux_c100` ;
- **H3 — proximité :** `theme_faux_c100` > `hors_theme_faux_c100` ;
- **H4 — même un graphe faux aide :** `hors_theme_faux_c100` > `aucun`.

**Comparaison directe des deux effets** (analyse secondaire) : effet de la couverture = expert 100 % − expert 25 % ; effet de la justesse = expert 100 % − theme_faux 100 % ;
on teste si le premier est plus grand que le second (différence, IC bootstrap, test unilatéral), pour GKT et, sur les mêmes graines, pour PrereqKT.

### Deux comptes Colab en parallèle
GKT est plus lent que PrereqKT. Le travail est coupé en deux moitiés **indépendantes** : `PARTIE = 'A'` (graines 0 à 4) et `PARTIE = 'B'` (graines 5 à 9).
Chaque moitié écrit **son propre fichier** : vous pouvez lancer A sur un compte et B sur l'autre **en même temps**, sans conflit. La cellule G4 réunit les deux.

**GPU T4 obligatoire.** Commencez par la cellule G2 (test rapide) : elle affiche le temps estimé.

In [ ]:
# Cellule 1 — Drive, GPU, paramètres
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import stats

DOSSIER   = '/content/drive/MyDrive/projet_kt'
PREP      = os.path.join(DOSSIER, 'junyi_prepare')
RESULTATS = os.path.join(DOSSIER, 'resultats')
os.makedirs(RESULTATS, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Appareil :', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(pas de GPU : ce sera lent)')

HP = dict(dim=64, hidden=128, couches_graphe=2, dropout=0.2, lr=1e-3, batch=64, epoques_max=60, patience=5)
GRAINE = 42

In [ ]:
# Cellule 2 — Séquences, graphe expert, et repérage des PREMIÈRES tentatives
exos = pd.read_csv(os.path.join(PREP, 'exercices.csv'))
N = len(exos)
A_expert = np.load(os.path.join(PREP, 'adjacence_expert.npy')).astype(np.float32)
aretes_expert = list(zip(*np.nonzero(A_expert)))

def charger(partie):
    df = pd.read_csv(os.path.join(PREP, f'sequences_{partie}.csv')).sort_values(['uid', 'morceau'])
    U, Q, R, F = [], [], [], []
    vus = {}
    for uid, ex, rep in zip(df['uid'], df['exercices'], df['reponses']):
        q = np.array(ex.split(','), dtype=np.int64)
        r = np.array(rep.split(','), dtype=np.int64)
        deja = vus.setdefault(uid, set())          # exercices déjà tentés par cet élève (tous morceaux confondus)
        f = np.zeros(len(q), dtype=bool)
        for i, x in enumerate(q):
            if x not in deja:
                f[i] = True
                deja.add(x)
        U.append(uid); Q.append(q); R.append(r); F.append(f)
    return np.array(U), Q, R, F

donnees = {p: charger(p) for p in ['train', 'valid', 'test']}
for p, (u, q, r, f) in donnees.items():
    n_rep = sum(map(len, q)); n_first = sum(int(x.sum()) for x in f)
    print(f'{p:5s} : {len(set(u)):6,} élèves | {n_rep:9,} réponses | premières tentatives : {n_first:8,} ({n_first/n_rep:.1%})')

In [ ]:
# Cellule 3 — Tenseurs (Q exercices, R réponses, M masque, F première tentative)
def tenseurs(q_list, r_list, f_list):
    L = max(map(len, q_list))
    Q = np.zeros((len(q_list), L), dtype=np.int64); R = np.zeros_like(Q)
    M = np.zeros((len(q_list), L), dtype=bool);     F = np.zeros_like(M)
    for i, (q, r, f) in enumerate(zip(q_list, r_list, f_list)):
        Q[i, :len(q)], R[i, :len(r)], M[i, :len(q)], F[i, :len(f)] = q, r, True, f
    return tuple(torch.from_numpy(x) for x in (Q, R, M, F))

def sous_ensemble_train(fraction, graine=GRAINE):
    u, q, r, f = donnees['train']
    if fraction >= 1:
        return q, r, f
    rng = np.random.default_rng(graine)
    eleves = np.unique(u)
    gardes = set(rng.choice(eleves, size=max(1, int(fraction * len(eleves))), replace=False))
    idx = [i for i, x in enumerate(u) if x in gardes]
    return [q[i] for i in idx], [r[i] for i in idx], [f[i] for i in idx]

T_valid = tenseurs(*donnees['valid'][1:])
T_test  = tenseurs(*donnees['test'][1:])
print('OK')

### Les modèles

- **GraphKT** : identique à l'étape 5 (le graphe lisse les représentations des exercices, avec un canal « prérequis » et un canal « successeurs »).
- **PrereqKT** : GraphKT **plus** 6 indicateurs calculés à chaque instant pour l'exercice à prédire :
  taux de réussite moyen de l'élève sur ses **prérequis** déjà tentés, part des prérequis déjà tentés, présence de prérequis, et les trois mêmes indicateurs pour ses **successeurs**.
  C'est la traduction directe de l'hypothèse « *s'il a échoué sur A, il échouera plus souvent sur B* ».
  Avec un graphe vide, ces indicateurs valent 0 : PrereqKT redevient GraphKT sans graphe.

In [ ]:
# Cellule 4 — Modèles
def normaliser_lignes(A):
    deg = A.sum(1, keepdims=True)
    return np.divide(A, deg, out=np.zeros_like(A), where=deg > 0)

class GraphKT(nn.Module):
    def __init__(self, n, hp, A, indicateurs=False):
        super().__init__()
        d, h = hp['dim'], hp['hidden']
        A = np.asarray(A, dtype=np.float32)
        self.register_buffer('P_pre', torch.from_numpy(normaliser_lignes(A.T.copy())))  # ligne j : prérequis de j
        self.register_buffer('P_suc', torch.from_numpy(normaliser_lignes(A.copy())))    # ligne i : successeurs de i
        self.n, self.indicateurs = n, indicateurs
        self.E = nn.Embedding(n, d)
        self.couches = nn.ModuleList([nn.ModuleDict({'pre': nn.Linear(d, d), 'suc': nn.Linear(d, d)})
                                      for _ in range(hp['couches_graphe'])])
        self.R = nn.Embedding(2, d)
        self.lstm = nn.LSTM(2 * d, h, batch_first=True)
        self.drop = nn.Dropout(hp['dropout'])
        n_ind = 6 if indicateurs else 0
        self.pred = nn.Sequential(nn.Linear(h + d + n_ind, h), nn.ReLU(), nn.Linear(h, 1))

    def representations(self):
        H = self.E.weight
        a_pre = (self.P_pre.sum(1, keepdim=True) > 0).float()
        a_suc = (self.P_suc.sum(1, keepdim=True) > 0).float()
        for c in self.couches:
            H = H + torch.relu(c['pre'](self.P_pre @ H)) * a_pre + torch.relu(c['suc'](self.P_suc @ H)) * a_suc
        return H

    def calcul_indicateurs(self, q, r):
        # Historique cumulé par exercice jusqu'à l'instant t : nombre d'essais et de réussites (B, L-1, N)
        B, L = q.shape
        essais = torch.zeros(B, L - 1, self.n, device=q.device)
        essais.scatter_(2, q[:, :-1].unsqueeze(-1), 1.0)
        reussites = essais * r[:, :-1].unsqueeze(-1).float()
        essais, reussites = essais.cumsum(1), reussites.cumsum(1)
        tente = (essais > 0).float()
        taux = reussites / essais.clamp(min=1)
        sortie = []
        for P in (self.P_pre, self.P_suc):
            voisins = (P[q[:, 1:]] > 0).float()                 # voisins de l'exercice à prédire (B, L-1, N)
            n_vois = voisins.sum(-1)
            n_tentes = (voisins * tente).sum(-1)
            moy = (voisins * tente * taux).sum(-1) / n_tentes.clamp(min=1)
            sortie += [moy, n_tentes / n_vois.clamp(min=1), (n_vois > 0).float()]
        return torch.stack(sortie, -1)

    def forward(self, q, r):
        H = self.representations()
        h, _ = self.lstm(torch.cat([H[q[:, :-1]], self.R(r[:, :-1])], dim=-1))
        x = [self.drop(h), H[q[:, 1:]]]
        if self.indicateurs:
            x.append(self.calcul_indicateurs(q, r))
        return self.pred(torch.cat(x, dim=-1)).squeeze(-1)

print('OK')

In [ ]:
# Cellule 5 — Graphes : expert, aucun, aléatoire, et corruption contrôlée E1–E5
import networkx as nx

def vers_matrice(aretes, n=N):
    A = np.zeros((n, n), dtype=np.float32)
    for a, b in aretes:
        A[a, b] = 1
    return A

def corrompre(aretes, type_err, taux, graine=GRAINE, topics=exos['topic'].values):
    rng = random.Random(10_000 + graine)
    E = set(map(tuple, aretes))
    k = int(round(taux * len(E)))
    noeuds = list(range(N))
    if type_err == 'E1':      # fausses flèches aléatoires
        while k > 0:
            a, b = rng.sample(noeuds, 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E2':    # fausses flèches plausibles : même thème
        par_topic = {}
        for i, t in enumerate(topics):
            par_topic.setdefault(t, []).append(i)
        groupes = [g for g in par_topic.values() if len(g) >= 2]
        essais = 0
        while k > 0 and essais < 100 * len(E) + 1000:
            essais += 1
            a, b = rng.sample(rng.choice(groupes), 2)
            if (a, b) not in E and (b, a) not in E:
                E.add((a, b)); k -= 1
    elif type_err == 'E3':    # inversions
        for a, b in rng.sample(sorted(E), min(k, len(E))):
            E.discard((a, b)); E.add((b, a))
    elif type_err == 'E4':    # suppressions
        for e in rng.sample(sorted(E), min(k, len(E))):
            E.discard(e)
    elif type_err == 'E5':    # cycles : C -> A alors que A mène déjà à C
        G = nx.DiGraph(list(E))
        candidats = sorted(E); rng.shuffle(candidats)
        for a, b in candidats:
            if k == 0:
                break
            c = rng.choice(list(nx.descendants(G, b)) + [b])
            if c != a and (c, a) not in E:
                E.add((c, a)); G.add_edge(c, a); k -= 1
    return sorted(E)

def graphe(nom, graine=GRAINE, taux=0.0):
    if nom == 'expert':
        return A_expert
    if nom == 'aucun':
        return np.zeros((N, N), dtype=np.float32)
    if nom == 'aleatoire':
        rng = random.Random(graine)
        E = set()
        while len(E) < len(aretes_expert):
            a, b = rng.sample(range(N), 2)
            E.add((a, b))
        return vers_matrice(E)
    return vers_matrice(corrompre(aretes_expert, nom, taux, graine))

def f1_aretes(A, B=A_expert):
    P, V = set(zip(*np.nonzero(A))), set(zip(*np.nonzero(B)))
    tp = len(P & V); p = tp / max(1, len(P)); r = tp / max(1, len(V))
    return round(2 * p * r / (p + r), 3) if p + r else 0.0

print(f'{"graphe":8s} {"taux":>5s} {"flèches":>8s} {"F1/expert":>10s} {"acyclique":>10s}')
for t in ['E1', 'E2', 'E3', 'E4', 'E5']:
    for taux in [0.1, 0.3, 0.5]:
        A = graphe(t, 0, taux)
        print(f'{t:8s} {taux:5.0%} {int(A.sum()):8d} {f1_aretes(A):10.3f} {str(nx.is_directed_acyclic_graph(nx.DiGraph(A))):>10s}')

In [ ]:
# Cellule 6 — Entraînement et évaluation par sous-ensemble
def fixer_graine(g):
    random.seed(g); np.random.seed(g); torch.manual_seed(g); torch.cuda.manual_seed_all(g)

@torch.no_grad()
def evaluer(modele, T, batch=1024):   # grands lots : même résultat, évaluation beaucoup plus rapide pour GKT
    modele.eval()
    Q, R, M, F = T
    ys, ps, fs = [], [], []
    for i in range(0, len(Q), batch):
        q, r, m, f = (x[i:i+batch].to(DEVICE) for x in (Q, R, M, F))
        p = torch.sigmoid(modele(q, r))
        c = m[:, 1:]
        ys.append(r[:, 1:][c].cpu().numpy()); ps.append(p[c].float().cpu().numpy()); fs.append(f[:, 1:][c].cpu().numpy())
    y, p, f = (np.concatenate(x) for x in (ys, ps, fs))
    auc = lambda s: round(float(roc_auc_score(y[s], p[s])), 4) if len(set(y[s])) == 2 else None
    tout = np.ones_like(f, dtype=bool)
    return {'AUC': auc(tout), 'AUC_1re': auc(f), 'AUC_rep': auc(~f), 'ACC': round(float(accuracy_score(y, p >= 0.5)), 4)}

def entrainer(nom_modele, A, fraction=1.0, graine=GRAINE, hp=HP):
    fixer_graine(graine)
    Q, R, M, F = tenseurs(*sous_ensemble_train(fraction, graine))
    modele = (GKT(N, hp, A) if nom_modele == 'GKT' else GraphKT(N, hp, A, indicateurs=(nom_modele == 'PrereqKT'))).to(DEVICE)
    opt = torch.optim.Adam(modele.parameters(), lr=hp['lr'])
    perte = nn.BCEWithLogitsLoss()
    meilleur, etat, attente, t0 = -1, None, 0, time.time()
    for ep in range(hp['epoques_max']):
        modele.train()
        ordre = torch.randperm(len(Q))
        for i in range(0, len(Q), hp['batch']):
            idx = ordre[i:i + hp['batch']]
            q, r, m = Q[idx].to(DEVICE), R[idx].to(DEVICE), M[idx].to(DEVICE)
            c = m[:, 1:]
            l = perte(modele(q, r)[c], r[:, 1:][c].float())
            opt.zero_grad(); l.backward()
            nn.utils.clip_grad_norm_(modele.parameters(), 5.0)
            opt.step()
        v = evaluer(modele, T_valid)['AUC']
        # print(f'   époque {ep+1:2d} | AUC valid {v:.4f} | {(time.time()-t0)/60:.1f} min')
        if v > meilleur + 1e-4:
            meilleur, etat, attente = v, copy.deepcopy(modele.state_dict()), 0
        else:
            attente += 1
            if attente >= hp['patience']:
                break
    modele.load_state_dict(etat)
    return {'AUC_valid': round(meilleur, 4), **{f'{k}_test': v for k, v in evaluer(modele, T_test).items()},
            'epoques': ep + 1, 'minutes': round((time.time() - t0) / 60, 1)}
print('OK')

In [ ]:
# Cellule F1 — Construction des 9 graphes factoriels (déterministe pour chaque graine)
COUVERTURES = [0.25, 0.5, 1.0]
SOURCES = ['expert', 'theme_faux', 'hors_theme_faux']
GRAINES = list(range(10))

G_exp = nx.DiGraph(); G_exp.add_nodes_from(range(N)); G_exp.add_edges_from(map(tuple, aretes_expert))
ancetres = {b: nx.ancestors(G_exp, b) for b in range(N)}
prereq_directs = {b: list(G_exp.predecessors(b)) for b in range(N)}
UNIVERS = sorted(b for b in range(N) if prereq_directs[b])
topic = exos.set_index('exo_id')['topic'].to_dict()
meme_theme = {}
for b in range(N):
    meme_theme[b] = [a for a in range(N) if a != b and isinstance(topic[b], str) and topic[a] == topic[b]]

def graphe_factoriel(couv, source, graine):
    rng = random.Random(90_000 + graine)
    ordre = UNIVERS[:]; rng.shuffle(ordre)                 # sous-ensembles emboîtés : 25 % ⊂ 50 % ⊂ 100 %
    choisis = ordre[:int(round(couv * len(ordre)))]
    E, manquants = [], 0
    for b in sorted(choisis):
        n_b = len(prereq_directs[b])
        if source == 'expert':
            E += [(a, b) for a in prereq_directs[b]]
            continue
        interdits = ancetres[b] | {b}
        if source == 'theme_faux':
            pool = [a for a in meme_theme[b] if a not in interdits]
        else:
            pool = [a for a in range(N) if a not in interdits and (not isinstance(topic[b], str) or topic[a] != topic[b])]
        rng_b = random.Random(f'{graine}-{source}-{b}')        # même tirage pour B à toutes les couvertures
        pris = rng_b.sample(pool, min(n_b, len(pool)))
        manquants += n_b - len(pris)
        E += [(a, b) for a in pris]
    return sorted(set(E)), manquants

# Vérifications : nombre de flèches, part des flèches entre exercices du même thème, couverture des premières tentatives
_, Qt, _, Ft = donnees['test']
premieres = pd.Series(np.concatenate([q[f] for q, f in zip(Qt, Ft)])).value_counts()
poids = premieres.reindex(range(N), fill_value=0).values
def couverture_1res(E):
    a = np.zeros(N, bool)
    for _, b in E:
        a[b] = True
    return 100 * (poids * a).sum() / poids.sum()

lignes = []
for couv in COUVERTURES:
    for source in SOURCES:
        m = []
        for g in GRAINES:
            E, manq = graphe_factoriel(couv, source, g)
            meme = np.mean([isinstance(topic[b], str) and topic[a] == topic[b] for a, b in E]) if E else 0
            juste = np.mean([a in ancetres[b] for a, b in E]) if E else 0
            m.append({'fleches': len(E), 'manquantes': manq, 'meme_theme_%': 100 * meme,
                      'prerequis_vrais_%': 100 * juste, 'couverture_1res_%': couverture_1res(E)})
        lignes.append({'couverture': couv, 'source': source, **pd.DataFrame(m).mean().round(1).to_dict()})
verif = pd.DataFrame(lignes)
verif.to_csv(os.path.join(RESULTATS, 'rq2_factoriel_graphes.csv'), index=False)
print(f'Univers : {len(UNIVERS)} exercices avec au moins un prérequis expert')
print(verif.to_string(index=False))

In [ ]:
# Cellule G1 — GKT (Nakagawa et al., 2019), variante à graphe fixe
HP_GKT = dict(HP, gkt_hidden=32, gkt_emb=32)

class GKT(nn.Module):
    def __init__(self, n, hp, A):
        super().__init__()
        dh, de = hp['gkt_hidden'], hp['gkt_emb']
        self.n, self.dh = n, dh
        A = np.asarray(A) > 0
        voisins = []
        for i in range(n):
            suc = [int(j) for j in np.nonzero(A[i])[0] if j != i]                  # i est prérequis de j
            vus = set(suc)
            pre = [int(j) for j in np.nonzero(A[:, i])[0] if j != i and j not in vus]   # j est prérequis de i
            voisins.append([(j, 1) for j in suc] + [(j, 2) for j in pre])
        K = max(1, max(map(len, voisins)))
        idx = np.full((n + 1, K), n, dtype=np.int64)        # n = nœud fictif servant de remplissage
        typ = np.zeros((n + 1, K), dtype=np.int64)          # 0 remplissage, 1 successeur, 2 prérequis
        for i, v in enumerate(voisins):
            for k, (j, t) in enumerate(v):
                idx[i, k], typ[i, k] = j, t
        self.register_buffer('NB', torch.from_numpy(idx))
        self.register_buffer('TY', torch.from_numpy(typ))
        self.h0 = nn.Parameter(torch.zeros(n + 1, dh))      # état initial appris de chaque exercice
        self.Ex = nn.Embedding(2 * n, de)                   # interaction (exercice, réponse)
        self.Ec = nn.Embedding(n + 1, de)                   # exercice
        mlp = lambda d_in: nn.Sequential(nn.Linear(d_in, dh), nn.ReLU(), nn.Dropout(hp['dropout']), nn.Linear(dh, dh))
        self.f_soi = mlp(dh + de)
        self.f_suc = mlp(2 * (dh + de))                     # message de i vers ses successeurs
        self.f_pre = mlp(2 * (dh + de))                     # message de i vers ses prérequis
        self.efface, self.ajoute = nn.Linear(dh, dh), nn.Linear(dh, dh)
        self.w_ea = nn.Parameter(torch.ones(n + 1))         # poids effacement-ajout propre à chaque exercice
        self.gru = nn.GRUCell(dh, dh)
        self.sortie = nn.Linear(dh, 1)

    def forward(self, q, r):
        B, L = q.shape
        h = self.h0.unsqueeze(0).expand(B, -1, -1).contiguous()
        lig = torch.arange(B, device=q.device)
        logits = []
        for t in range(L - 1):
            i = q[:, t]
            h_i = h[lig, i]
            soi = torch.cat([h_i, self.Ex(2 * i + r[:, t])], -1)
            nb, ty = self.NB[i], self.TY[i]                                     # (B, K)
            h_nb = torch.gather(h, 1, nb.unsqueeze(-1).expand(-1, -1, self.dh))
            paire = torch.cat([soi.unsqueeze(1).expand(-1, nb.size(1), -1), h_nb, self.Ec(nb)], -1)
            m_nb = torch.where((ty == 1).unsqueeze(-1), self.f_suc(paire), self.f_pre(paire))
            m = torch.cat([self.f_soi(soi).unsqueeze(1), m_nb], 1)              # (B, K+1, dh)
            cible = torch.cat([i.unsqueeze(1), nb], 1)
            w = self.w_ea[cible].unsqueeze(-1)
            m = m - w * torch.sigmoid(self.efface(m)) * m + w * torch.tanh(self.ajoute(m))
            h_old = torch.cat([h_i.unsqueeze(1), h_nb], 1)
            h_new = self.gru(m.reshape(-1, self.dh), h_old.reshape(-1, self.dh)).view_as(h_old)
            valide = torch.cat([torch.ones_like(ty[:, :1], dtype=torch.bool), ty > 0], 1).unsqueeze(-1)
            h_new = torch.where(valide, h_new, h_old)
            h = h.scatter(1, cible.unsqueeze(-1).expand(-1, -1, self.dh), h_new)
            logits.append(self.sortie(h[lig, q[:, t + 1]]).squeeze(-1))
        return torch.stack(logits, 1)

print('GKT prêt. Plus grand nombre de voisins (graphe expert) :', GKT(N, HP_GKT, A_expert).NB.shape[1])

In [ ]:
# Cellule G2 — Test rapide : 2 époques sur le graphe expert (vérifie la mémoire et estime la durée totale)
HP_TEST = dict(HP_GKT, epoques_max=2, patience=99)
for frac in [0.05, 0.01]:
    r_ = entrainer('GKT', A_expert, fraction=frac, graine=0, hp=HP_TEST)
    print(f'{frac:.0%} des élèves : {r_["minutes"] / 2:.2f} min par époque | AUC 1res tentatives après 2 époques = {r_["AUC_1re_test"]}')
if DEVICE == 'cuda':
    print(f'Mémoire GPU maximale : {torch.cuda.max_memory_allocated() / 1e9:.1f} Go')
print("Estimation : environ 25 époques par entraînement ; multipliez le temps par époque par 25, puis par 25 entraînements par régime et par moitié.")

In [ ]:
# Cellule G3 — Les entraînements de votre moitié, graine par graine (reprise automatique)
PARTIE = 'B1'         # ce notebook : graines 5, 6 et 7

GRAINES_PARTIE = {'A': [0, 1, 2, 3, 4], 'B': [5, 6, 7, 8, 9], 'B1': [5, 6, 7], 'B2': [8, 9]}[PARTIE]
PARTIES = ['A', 'B', 'B1', 'B2']      # fichiers rq2_gkt_<partie>.csv lus pour la reprise et l'analyse
FRACTIONS = [0.05, 0.01]
CONDITIONS_GKT = [('aucun', None, None), ('expert_c25', 0.25, 'expert'), ('expert_c100', 1.0, 'expert'),
                  ('theme_faux_c100', 1.0, 'theme_faux'), ('hors_theme_faux_c100', 1.0, 'hors_theme_faux')]
chemin_g = os.path.join(RESULTATS, f'rq2_gkt_{PARTIE}.csv')
res_g = pd.read_csv(chemin_g).to_dict('records') if os.path.exists(chemin_g) else []
fait = set()                        # tout ce qui est déjà fait, quelle que soit la partie qui l'a fait
for p_ in PARTIES:
    c_ = os.path.join(RESULTATS, f'rq2_gkt_{p_}.csv')
    if os.path.exists(c_):
        fait |= {(x['condition'], float(x['fraction']), int(x['graine'])) for x in pd.read_csv(c_).to_dict('records')}
t0_ = time.time()
for g_ in GRAINES_PARTIE:
    for frac in FRACTIONS:
        for lib, couv, source in CONDITIONS_GKT:
            if (lib, frac, g_) in fait:
                continue
            A = graphe('aucun') if couv is None else vers_matrice(graphe_factoriel(couv, source, g_)[0])
            res = entrainer('GKT', A, fraction=frac, graine=g_, hp=HP_GKT)
            res_g.append({'modele': 'GKT', 'condition': lib, 'couverture': couv or 0.0, 'source': source or 'aucun',
                          'fraction': frac, 'graine': g_, **res})
            pd.DataFrame(res_g).to_csv(chemin_g, index=False)
            print(f'  graine {g_} | {frac:.0%} | {lib:22s} | AUC 1res = {res["AUC_1re_test"]} | {res["epoques"]} époques | {res["minutes"]} min')
    print(f'Graine {g_} terminée ({(time.time() - t0_) / 60:.0f} min écoulées)')
print(f'Moitié {PARTIE} terminée.')

In [ ]:
# Cellule G4 — Analyse (à lancer quand les deux moitiés sont finies ; marche aussi avec les graines déjà terminées)
#   Après une déconnexion, il suffit de relancer la cellule 1 puis celle-ci.
import matplotlib.pyplot as plt
col = 'AUC_1re_test'
FRACTIONS = [0.05, 0.01]
LIBS = ['aucun', 'expert_c25', 'expert_c100', 'theme_faux_c100', 'hors_theme_faux_c100']
parts = [pd.read_csv(os.path.join(RESULTATS, f'rq2_gkt_{p}.csv')) for p in ['A', 'B', 'B1', 'B2']
         if os.path.exists(os.path.join(RESULTATS, f'rq2_gkt_{p}.csv'))]
gkt = pd.concat(parts, ignore_index=True).drop_duplicates(['condition', 'fraction', 'graine'])
n_par_graine = gkt.groupby('graine').size()
graines_ok = sorted(n_par_graine[n_par_graine == len(LIBS) * len(FRACTIONS)].index)
gkt = gkt[gkt.graine.isin(graines_ok)]
print(f'Graines complètes analysées : {graines_ok}')

pkt = pd.read_csv(os.path.join(RESULTATS, 'rq2_factoriel.csv'))           # PrereqKT, étape 13, mêmes graines
pkt = pkt[pkt.condition.isin(LIBS) & pkt.graine.isin(graines_ok)]

def ic_bootstrap(d, n=10_000, graine=0):
    rng = np.random.default_rng(graine); d = np.asarray(d)
    return np.percentile(rng.choice(d, size=(n, len(d)), replace=True).mean(1), [2.5, 97.5])

synthese, hyp, rapports = [], [], []
for nom, tab in [('GKT', gkt), ('PrereqKT', pkt)]:
    for frac in FRACTIONS:
        piv = tab[tab.fraction == frac].pivot_table(index='graine', columns='condition', values=col)
        base, gain_exp = piv['aucun'], piv['expert_c100'] - piv['aucun']
        for lib in LIBS:
            d = piv[lib] - base
            lo, hi = ic_bootstrap(d) if lib != 'aucun' else (0, 0)
            synthese.append({'modele': nom, 'fraction': frac, 'condition': lib, 'AUC_1re': round(piv[lib].mean(), 4),
                             'gain_vs_aucun': round(d.mean(), 4), 'IC95_bas': round(lo, 4), 'IC95_haut': round(hi, 4),
                             'benefice_conserve_%': round(100 * d.mean() / gain_exp.mean(), 1)})
        # Effets orientés dans le sens de l'effet du graphe pour ce modèle (gain pour PrereqKT, perte pour GKT) :
        # on compare ainsi la TAILLE de l'effet de la couverture et de celui de la justesse.
        sens = np.sign(gain_exp.mean()) or 1.0
        cov = sens * (piv['expert_c100'] - piv['expert_c25']); jus = sens * (piv['expert_c100'] - piv['theme_faux_c100'])
        rng = np.random.default_rng(1); idx = rng.integers(0, len(cov), size=(10_000, len(cov)))
        bs = (cov - jus).values[idx].mean(1)
        rapports.append({'modele': nom, 'fraction': frac, 'sens_effet_graphe': 'gain' if sens > 0 else 'perte', 'effet_couverture': round(cov.mean(), 4), 'effet_justesse': round(jus.mean(), 4),
                         'couverture_moins_justesse': round((cov - jus).mean(), 4),
                         'IC95_bas': round(np.percentile(bs, 2.5), 4), 'IC95_haut': round(np.percentile(bs, 97.5), 4),
                         'graines_couverture_plus_forte': f'{int(((cov - jus) > 0).sum())}/{len(cov)}',
                         'p_unilateral': round(stats.ttest_1samp(cov - jus, 0, alternative="greater").pvalue, 4),
                         'rapport_couverture_sur_justesse': round(cov.mean() / jus.mean(), 1) if jus.mean() > 0 else None})
        if nom == 'GKT':
            for h_, a, b in [('H1 couverture', 'expert_c100', 'expert_c25'), ('H2 justesse', 'expert_c100', 'theme_faux_c100'),
                             ('H3 proximité', 'theme_faux_c100', 'hors_theme_faux_c100'), ('H4 un graphe faux aide', 'hors_theme_faux_c100', 'aucun')]:
                d = piv[a] - piv[b]
                hyp.append({'fraction': frac, 'hypothese': f'{h_} : {a} > {b}', 'ecart_moyen': round(d.mean(), 4),
                            'gagne': f'{int((d > 0).sum())}/{len(d)}', 'p': stats.ttest_1samp(d, 0, alternative='greater').pvalue})

synthese, hyp, rapports = pd.DataFrame(synthese), pd.DataFrame(hyp), pd.DataFrame(rapports)
o = hyp['p'].sort_values(); m = len(o)
hyp['p_holm'] = (o * (m - np.arange(m))).cummax().clip(upper=1).reindex(hyp.index)
hyp['confirmee_holm_5%'] = hyp['p_holm'] < 0.05
hyp[['p', 'p_holm']] = hyp[['p', 'p_holm']].round(4)
synthese.to_csv(os.path.join(RESULTATS, 'rq2_gkt_synthese.csv'), index=False)
hyp.to_csv(os.path.join(RESULTATS, 'rq2_gkt_hypotheses.csv'), index=False)
rapports.to_csv(os.path.join(RESULTATS, 'rq2_gkt_rapports.csv'), index=False)
print('\n===== Gain d\'AUC (1res tentatives) par rapport à « aucun », GKT et PrereqKT =====')
print(synthese.to_string(index=False))
print('\n===== Hypothèses principales GKT (Holm sur 8 tests) =====')
print(hyp.to_string(index=False))
print('\n===== Taille de l\'effet de la couverture comparée à celle de la justesse (effets orientés dans le sens de l\'effet du graphe) =====')
print(rapports.to_string(index=False))

# Figure : gain d'AUC (1res tentatives) par rapport à « aucun », GKT à côté de PrereqKT
#   (en points d'AUC et non en % : avec GKT, le gain du graphe expert peut être négatif)
NOMS = {'expert_c25': 'expert\n25 %', 'expert_c100': 'expert\n100 %', 'theme_faux_c100': 'faux même\nthème 100 %',
        'hors_theme_faux_c100': 'faux autre\nthème 100 %'}
fig, axes = plt.subplots(1, 2, figsize=(10, 3.9), sharey=True)
for ax, frac in zip(axes, FRACTIONS):
    x = np.arange(len(NOMS))
    for nom, dec in [('PrereqKT', -0.2), ('GKT', 0.2)]:
        s = synthese[(synthese.modele == nom) & (synthese.fraction == frac)].set_index('condition').loc[list(NOMS)]
        err = np.vstack([s.gain_vs_aucun - s.IC95_bas, s.IC95_haut - s.gain_vs_aucun]).clip(min=0)
        ax.bar(x + dec, s['gain_vs_aucun'], width=0.4, yerr=err, capsize=3, label=nom)
    ax.set_xticks(x); ax.set_xticklabels(NOMS.values(), fontsize=8)
    ax.axhline(0, c='black', lw=0.8); ax.set_title(f'{frac:.0%} des élèves')
axes[0].set_ylabel("Gain d'AUC sur les 1res tentatives\n(par rapport à « aucun graphe »)")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(RESULTATS, 'rq2_gkt.png'), dpi=200)
plt.show()

### Contrôles (à lancer après les deux moitiés A et B)

- **G5 — GKT a-t-il fini d'apprendre ?** À 1 % des élèves, presque tous les entraînements GKT s'arrêtent à la limite de 60 époques.
  On refait la graine 0, conditions `aucun` et `expert_c100`, avec une limite de **200 époques** (tout le reste identique). Si l'écart entre les deux reste négatif, la baisse due au graphe n'est pas un manque d'entraînement. Environ 40 min de GPU.
- **G6 — Référence « difficulté seule ».** On prédit chaque première tentative par le **taux de réussite de l'exercice** chez les élèves d'entraînement (lissé). Aucun réseau, aucun graphe, pas de GPU, quelques minutes.
  Cela situe tous les modèles par rapport à ce qu'apporte la seule difficulté des exercices.

Après une déconnexion : relancez les cellules 1 à G1 (pour G5) ou 1 à 3 (pour G6), puis la cellule voulue.

In [ ]:
# Cellule G5 — Contrôle de convergence : GKT, 1 % des élèves, graine 0, jusqu'à 200 époques
HP_LONG = dict(HP_GKT, epoques_max=200)
chemin_c = os.path.join(RESULTATS, 'rq2_gkt_controle_epoques.csv')
res_c = pd.read_csv(chemin_c).to_dict('records') if os.path.exists(chemin_c) else []
fait_c = {x['condition'] for x in res_c}
for lib, couv, source in [('aucun', None, None), ('expert_c100', 1.0, 'expert')]:
    if lib in fait_c:
        continue
    A = graphe('aucun') if couv is None else vers_matrice(graphe_factoriel(couv, source, 0)[0])
    res = entrainer('GKT', A, fraction=0.01, graine=0, hp=HP_LONG)
    res_c.append({'condition': lib, 'fraction': 0.01, 'graine': 0, 'epoques_max': 200, **res})
    pd.DataFrame(res_c).to_csv(chemin_c, index=False)
    print(f'{lib:12s} | AUC 1res = {res["AUC_1re_test"]} | {res["epoques"]} époques | {res["minutes"]} min')

avant = pd.read_csv(os.path.join(RESULTATS, 'rq2_gkt_A.csv'))
avant = avant[(avant.graine == 0) & (avant.fraction == 0.01)].set_index('condition')
apres = pd.DataFrame(res_c).set_index('condition')
comp = pd.DataFrame({'AUC_1re_60_epoques': avant.loc[['aucun', 'expert_c100'], 'AUC_1re_test'],
                     'epoques_utilisees_60': avant.loc[['aucun', 'expert_c100'], 'epoques'],
                     'AUC_1re_200_epoques': apres.loc[['aucun', 'expert_c100'], 'AUC_1re_test'],
                     'epoques_utilisees_200': apres.loc[['aucun', 'expert_c100'], 'epoques']})
print(comp.to_string())
print(f"\nÉcart expert − aucun : {comp.at['expert_c100', 'AUC_1re_60_epoques'] - comp.at['aucun', 'AUC_1re_60_epoques']:+.4f} (60 époques) | "
      f"{comp.at['expert_c100', 'AUC_1re_200_epoques'] - comp.at['aucun', 'AUC_1re_200_epoques']:+.4f} (200 époques)")

In [ ]:
# Cellule G6 — Référence « difficulté seule » : taux de réussite lissé de chaque exercice (sans GPU)
LISSAGE = 5        # équivaut à 5 réponses fictives au taux de réussite moyen
_, Qt, Rt, Ft = donnees['test']
q_te = np.concatenate([q[1:] for q in Qt]); r_te = np.concatenate([r[1:] for r in Rt]); f_te = np.concatenate([f[1:] for f in Ft])
# (on ignore la 1re réponse de chaque séquence, comme les modèles, qui ne la prédisent pas)

lignes = []
for frac in [0.05, 0.01]:
    for g_ in range(10):
        q_tr, r_tr, f_tr = sous_ensemble_train(frac, g_)
        q_all, r_all, f_all = np.concatenate(q_tr), np.concatenate(r_tr), np.concatenate(f_tr)
        ligne = {'fraction': frac, 'graine': g_}
        for nom_v, garde in [('toutes_tentatives', np.ones_like(f_all, bool)), ('premieres_tentatives', f_all)]:
            n_ = np.bincount(q_all[garde], minlength=N); s_ = np.bincount(q_all[garde], weights=r_all[garde], minlength=N)
            moy = s_.sum() / n_.sum()
            taux = (s_ + LISSAGE * moy) / (n_ + LISSAGE)
            p_ = taux[q_te]
            ligne[f'AUC_1re_{nom_v}'] = round(roc_auc_score(r_te[f_te], p_[f_te]), 4)
            ligne[f'AUC_tout_{nom_v}'] = round(roc_auc_score(r_te, p_), 4)
        lignes.append(ligne)
ref = pd.DataFrame(lignes)
ref.to_csv(os.path.join(RESULTATS, 'rq2_reference_difficulte.csv'), index=False)

# Comparaison avec les modèles (mêmes graines, AUC des 1res tentatives)
gk = pd.concat([pd.read_csv(os.path.join(RESULTATS, f'rq2_gkt_{p}.csv')) for p in ['A', 'B', 'B1', 'B2']
                if os.path.exists(os.path.join(RESULTATS, f'rq2_gkt_{p}.csv'))]).drop_duplicates(['condition', 'fraction', 'graine'])
pk = pd.read_csv(os.path.join(RESULTATS, 'rq2_factoriel.csv'))
tab = []
for frac in [0.05, 0.01]:
    r_ = ref[ref.fraction == frac]
    tab.append({'fraction': frac, 'modele': 'difficulté seule (toutes tentatives)', 'AUC_1re': r_['AUC_1re_toutes_tentatives'].mean()})
    tab.append({'fraction': frac, 'modele': 'difficulté seule (1res tentatives)', 'AUC_1re': r_['AUC_1re_premieres_tentatives'].mean()})
    for nom, t in [('GKT', gk), ('PrereqKT', pk)]:
        for lib in ['aucun', 'expert_c100']:
            v = t[(t.fraction == frac) & (t.condition == lib)]['AUC_1re_test']
            tab.append({'fraction': frac, 'modele': f'{nom} {lib}', 'AUC_1re': v.mean(), 'graines': len(v)})
tab = pd.DataFrame(tab).round(4)
tab.to_csv(os.path.join(RESULTATS, 'rq2_reference_comparaison.csv'), index=False)
print(tab.to_string(index=False))